# Lab Exercise: Missing Value Imputation
This notebook explores the statistical implications of missing data and compares classical imputation methods (Mean, Median) against multivariate solvers (KNN, Iterative).

We will use California Housing data and evaluate imputer accuracy by surgically hiding coordinates we already possess.

In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

warnings.filterwarnings('ignore')
rng = np.random.RandomState(0)

# Sample a subset of the housing data
df = fetch_california_housing(as_frame=True).frame.sample(2000, random_state=0).reset_index(drop=True)
truth = df['MedInc']

print(f'Loaded {len(df)} sample rows from California Housing.')

### 1. MCAR vs. MNAR and Mean Imputation
We compare how MCAR (random drop) vs MNAR (non-random drops, e.g., missing because salary is too high) alter distribution statistics after mean imputation.

In [ ]:
print('WHAT MEAN IMPUTATION DOES TO A DISTRIBUTION')
print(f"   {'scenario':<36}{'mean':>8}{'std':>8}{'corr':>9}")

def report(label, col):
    print(f"   {label:<36}{col.mean():>8.3f}{col.std():>8.3f}{col.corr(df['HouseAge']):>+9.3f}")

report('complete truth', truth)

mcar = truth.copy()
mcar[rng.rand(len(mcar)) < 0.30] = np.nan  # Missing completely at random
report('MCAR 30% gone, before imputing', mcar)
report('MCAR + mean imputation', mcar.fillna(mcar.mean()))

mnar = truth.copy()
mnar[truth > truth.quantile(0.70)] = np.nan  # MNAR: top earners missing
report('MNAR top 30% hidden, before', mnar)
report('MNAR + mean imputation', mnar.fillna(mnar.mean()))

### 2. Deletion Arithmetic
Trace how listwise deletion (`dropna`) destroys row volumes as column dimensions scale up.

In [ ]:
X = df.drop(columns=['MedHouseVal'])
print(f'LISTWISE DELETION — dropna() on {X.shape[1]} columns')
print(f"   {'missing per column':>20}{'rows kept (theory)':>20}{'measured':>10}")
for p in [0.01, 0.05, 0.10]:
    theory = (1 - p) ** X.shape[1]
    measured = X.mask(rng.rand(*X.shape) < p).dropna().shape[0] / len(X)
    print(f'   {p:>19.0%}{theory:>19.1%}{measured:>10.1%}')

### 3. Comparing Imputers (Smarter Solvers)
We evaluate KNN and Iterative (MICE) imputer performances against simple mean fills, measuring Mean Absolute Error (MAE).

In [ ]:
def trial(cols, label):
    Xc = df[cols].copy()
    local_rng = np.random.RandomState(42)
    
    # Hide 20% of values in the target column
    target_col = cols[0]
    gaps = local_rng.rand(len(Xc)) < 0.20
    truth_val = Xc.loc[gaps, target_col]
    
    Xc.loc[gaps, target_col] = np.nan
    
    # 1. Mean fill
    f_mean = Xc[target_col].fillna(Xc[target_col].mean()).loc[gaps]
    err_mean = np.abs(truth_val - f_mean).mean()
    
    # 2. KNN imputer
    knn_imp = KNNImputer(n_neighbors=5)
    f_knn = pd.DataFrame(knn_imp.fit_transform(Xc), columns=cols).loc[gaps, target_col]
    err_knn = np.abs(truth_val - f_knn).mean()
    
    # 3. Iterative imputer
    iter_imp = IterativeImputer(random_state=0)
    f_iter = pd.DataFrame(iter_imp.fit_transform(Xc), columns=cols).loc[gaps, target_col]
    err_iter = np.abs(truth_val - f_iter).mean()
    
    print(f'Trial on {label} (r = {df[cols[0]].corr(df[cols[1]]):.2f})')
    print(f'   MAE - Mean:      {err_mean:.4f}')
    print(f'   MAE - KNN:       {err_knn:.4f}')
    print(f'   MAE - Iterative: {err_iter:.4f}\n')

# High correlation trial
trial(['AveRooms', 'AveBedrms'], 'AveRooms (predicting from AveBedrms)')

# Low correlation trial
trial(['MedInc', 'HouseAge'], 'MedInc (predicting from HouseAge)')